# CIFAR-10 Image Classification using ANN and CNN

## Assignment Objective
Build an image classification model on the CIFAR-10 dataset and analyze performance across different architectures and training strategies.

In this notebook, I trained and compared:

1. **ANN (Artificial Neural Network)**
2. **Basic CNN (Convolutional Neural Network)**
3. **Improved CNN with Batch Normalization and Dropout**
4. **CNN with Data Augmentation and Early Stopping**

The final analysis compares accuracy, loss, confusion matrix, classification report, and model behavior.

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow Version:", tf.__version__)

## 2. Load CIFAR-10 Dataset

CIFAR-10 contains **60,000 color images** of size **32 × 32 × 3**.

- 50,000 training images
- 10,000 test images
- 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)
print("Training labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

## 3. Visualize Sample Images

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[int(y_train[i])])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

Pixel values are originally in the range **0 to 255**.  
We normalize them to **0 to 1** so the model trains faster and more smoothly.

For ANN, images are flattened into vectors.  
For CNN, original image shape is preserved.

In [ ]:
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm = x_test.astype('float32') / 255.0

# ANN needs flat input: 32*32*3 = 3072 features
x_train_flat = x_train_norm.reshape(x_train_norm.shape[0], -1)
x_test_flat = x_test_norm.reshape(x_test_norm.shape[0], -1)

print("ANN train shape:", x_train_flat.shape)
print("CNN train shape:", x_train_norm.shape)

## 5. Helper Functions

These functions are used for plotting learning curves and evaluating models.

In [ ]:
def plot_history(history, title):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(title + ' - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(title + ' - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()


def evaluate_model(model, x_test_data, y_test_data, model_name):
    test_loss, test_accuracy = model.evaluate(x_test_data, y_test_data, verbose=0)
    print(f"{model_name} Test Loss: {test_loss:.4f}")
    print(f"{model_name} Test Accuracy: {test_accuracy:.4f}")
    return test_loss, test_accuracy

## 6. Model 1: ANN Architecture

ANN treats each image as a flat vector of 3072 values.  
It can classify images, but it does not understand spatial relationships like edges, shapes, and textures.

In [ ]:
ann_model = models.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

In [ ]:
ann_history = ann_model.fit(
    x_train_flat,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

ann_loss, ann_acc = evaluate_model(ann_model, x_test_flat, y_test, "ANN")
plot_history(ann_history, "ANN")

## 7. Model 2: Basic CNN Architecture

CNN is better for images because it keeps the 2D structure of images and learns local patterns using convolution filters.

In [ ]:
basic_cnn_model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

basic_cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

basic_cnn_model.summary()

In [ ]:
basic_cnn_history = basic_cnn_model.fit(
    x_train_norm,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

basic_cnn_loss, basic_cnn_acc = evaluate_model(basic_cnn_model, x_test_norm, y_test, "Basic CNN")
plot_history(basic_cnn_history, "Basic CNN")

## 8. Model 3: Improved CNN with Batch Normalization and Dropout

Training strategies used:

- **Batch Normalization:** stabilizes and speeds up training
- **Dropout:** reduces overfitting
- **More convolution layers:** improves feature extraction

In [ ]:
improved_cnn_model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.30),

    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.40),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.50),
    layers.Dense(10, activation='softmax')
])

improved_cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

improved_cnn_model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

improved_cnn_history = improved_cnn_model.fit(
    x_train_norm,
    y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

improved_cnn_loss, improved_cnn_acc = evaluate_model(improved_cnn_model, x_test_norm, y_test, "Improved CNN")
plot_history(improved_cnn_history, "Improved CNN")

## 9. Model 4: CNN with Data Augmentation

Data augmentation creates slightly modified images during training.  
This improves generalization because the model sees different versions of the same image.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

aug_cnn_model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,

    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

aug_cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

aug_cnn_model.summary()

In [ ]:
aug_history = aug_cnn_model.fit(
    x_train_norm,
    y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

aug_cnn_loss, aug_cnn_acc = evaluate_model(aug_cnn_model, x_test_norm, y_test, "Augmented CNN")
plot_history(aug_history, "Augmented CNN")

## 10. Model Performance Comparison

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['ANN', 'Basic CNN', 'Improved CNN', 'Augmented CNN'],
    'Test Loss': [ann_loss, basic_cnn_loss, improved_cnn_loss, aug_cnn_loss],
    'Test Accuracy': [ann_acc, basic_cnn_acc, improved_cnn_acc, aug_cnn_acc]
})

comparison_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(comparison_df['Model'], comparison_df['Test Accuracy'])
plt.title('Model Accuracy Comparison')
plt.xlabel('Model')
plt.ylabel('Test Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()

## 11. Confusion Matrix and Classification Report

The best model is selected using test accuracy, then we analyze its class-wise performance.

In [ ]:
models_dict = {
    'ANN': (ann_model, x_test_flat),
    'Basic CNN': (basic_cnn_model, x_test_norm),
    'Improved CNN': (improved_cnn_model, x_test_norm),
    'Augmented CNN': (aug_cnn_model, x_test_norm)
}

best_model_name = comparison_df.sort_values('Test Accuracy', ascending=False).iloc[0]['Model']
best_model, best_x_test = models_dict[best_model_name]

print("Best Model:", best_model_name)

y_pred_prob = best_model.predict(best_x_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test.flatten()

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title('Confusion Matrix - ' + best_model_name)
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.xticks(range(10), class_names, rotation=45)
plt.yticks(range(10), class_names)
plt.colorbar()
plt.tight_layout()
plt.show()

## 12. Prediction on Sample Test Images

In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    image = x_test_norm[i]

    if best_model_name == 'ANN':
        pred = best_model.predict(image.reshape(1, -1), verbose=0)
    else:
        pred = best_model.predict(image.reshape(1, 32, 32, 3), verbose=0)

    predicted_class = class_names[np.argmax(pred)]
    actual_class = class_names[int(y_test[i])]

    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[i])
    plt.title(f"A: {actual_class}
P: {predicted_class}")
    plt.axis('off')

plt.tight_layout()
plt.show()

## 13. Final Analysis

### ANN vs CNN
ANN uses flattened image pixels, so it loses spatial information. Because of this, ANN usually gives lower accuracy on image datasets.

CNN performs better because it uses convolution layers to detect edges, textures, shapes, and object patterns from images.

### Effect of Training Strategies
- **Dropout** helps reduce overfitting.
- **Batch Normalization** helps stabilize and speed up training.
- **Data Augmentation** improves generalization by creating modified versions of training images.
- **Early Stopping** prevents unnecessary training when validation performance stops improving.

### Overall Observation
The CNN-based models perform better than ANN because CIFAR-10 is an image dataset and CNN is designed for image feature extraction.

## 14. Conclusion

In this assignment, I built image classification models on the CIFAR-10 dataset using both ANN and CNN. The ANN model worked as a baseline, but it was limited because it treated images as flat vectors. CNN models performed better because they preserved image structure and learned spatial features.

After comparing different architectures and training strategies, I observed that CNN with regularization techniques like dropout, batch normalization, augmentation, and early stopping gives better generalization and improved performance.